# Kids art v3: realistic 3D picture boards for the fun-facts series (The CurioKids Club)

* **18 pictures**, one key visual per fun-facts episode that has no good picture yet (e.g. ep081: a child blinking), **landscape**, **2 versions** each (36 pictures, ~25 min on a T4).
* Style: soft, realistic 3D animated-family-film look (like the owner's references), bright and friendly. Some show a cute 3D child; no text or logos.
* Model: **FLUX.1-schnell** (Apache 2.0). Output: **art_kids_v3.zip**.

**Settings:** GPU **T4 x2**, Internet **On**, Add-ons → Secrets → **HF_TOKEN** ticked.
**Run:** **Save Version → Save & Run All (Commit)** (keeps the output; you can close the tab) → when done, version → **Output** → download **art_kids_v3.zip**.
If you run it by hand instead: **download the zip before stopping**. A numpy `_slice` error means: stop the session (power button) and start a fresh one.

In [ ]:
# 1) Install the libraries (about 2 minutes). Kaggle's own numpy version is kept: a newer numpy breaks the model loader.
import numpy; NP = numpy.__version__; print("keeping numpy", NP)
!pip install -q -U "diffusers>=0.32" "transformers>=4.46" accelerate bitsandbytes sentencepiece "numpy=={NP}"

In [ ]:
# 2) Check the GPU
import torch, subprocess
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU T4 x2 (Session options, right-hand panel) and run again."
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "GB")
# T4/P100 have no fast bfloat16, so they use float16; newer GPUs use bfloat16
DTYPE = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
print("compute type:", DTYPE)

In [ ]:
# 3) The pictures (name, what to show). Keep "no text" in every prompt.
KID3D = ("realistic 3D render in the style of a modern animated family film, soft rounded shapes, bright cheerful colours, "
         "soft warm cinematic lighting, gentle depth of field, high detail, friendly and safe for children, "
         "no text, no letters, no numbers, no logos, no watermark")
CHILD = "a cute 3D animated-film-style child about 7 years old, friendly face, "
L = "landscape"
PICTURES = [(n, w, KID3D, [(L, 11), (L, 42)]) for n, w in [
  ("kf_blink",        CHILD + "close-up of the face in the middle of a blink, one eye closing, long eyelashes, a tiny sparkle of tear film"),
  ("kf_space_cold",   "an astronaut in a white suit floating in dark cold space, frost crystals on the visor, Earth small and far away"),
  ("kf_plants_soil",  "two cut-away flower pots side by side: a cactus in dry sandy soil and a fern in dark moist soil, roots visible"),
  ("kf_brushing",     CHILD + "brushing teeth in front of a bathroom mirror with fluffy white toothpaste foam, smiling"),
  ("kf_star_glow",    "a glowing star close-up with swirling golden plasma and soft rays in deep blue space"),
  ("kf_wrinkly",      CHILD + "close-up of small wrinkly fingertips after a bath, soap bubbles around, warm bathroom light"),
  ("kf_straw_bent",   "side view of a clear glass of water with a red and white striped straw that looks broken and shifted exactly at the water surface, refraction"),
  ("kf_tummy",        "a friendly 3D model of a stomach with little bubbles and soft sound-wave rings around it, playful"),
  ("kf_apple_brown",  "two apple halves on a wooden board: the left half fresh and white, the right half clearly turned brown"),
  ("kf_shiver",       CHILD + "wrapped in a red scarf in the snow, shivering, tiny goosebumps, breath visible in the cold air"),
  ("kf_touch",        CHILD + "hand gently touching a soft white feather next to a rough brown pine cone, close-up"),
  ("kf_cloud_drops",  "a fluffy white cloud with a magnified circle showing it is made of tiny water droplets, sunny blue sky"),
  ("kf_dizzy",        CHILD + "spinning happily on a playground roundabout, motion blur on the background, giggling"),
  ("kf_shadows",      "a small toy figure on a table lit by a torch, casting one long shadow and one short shadow"),
  ("kf_eyebrows",     CHILD + "close-up of the face with raised eyebrows, a drop of water rolling around the eyebrow instead of into the eye"),
  ("kf_metal_wood",   "a shiny metal spoon and a wooden spoon side by side on a table, a frosty cold sparkle on the metal one"),
  ("kf_rumble_food",  "a plate with an apple, bread and a glass of milk on a sunny kitchen table, cosy"),
  ("kf_blink_eye",    "a close-up of a big friendly 3D animated-film-style eye with long eyelashes, half closed in a blink, a shiny tear film"),
]]
SIZES = {L: (1536, 864, (1920, 1080))}
TOTAL = sum(len(v) for *_, v in PICTURES)
print(TOTAL, "pictures to make")

In [ ]:
# 4) Load FLUX.1-schnell in 4-bit so it fits on one 16 GB GPU (first time downloads ~33 GB; about 5-10 minutes)
import os, gc
from diffusers import FluxPipeline, FluxTransformer2DModel, BitsAndBytesConfig as DiffBnb
from transformers import T5EncoderModel, BitsAndBytesConfig as TfBnb
MODEL = "black-forest-labs/FLUX.1-schnell"
# Needs the HF_TOKEN Kaggle secret (see the steps at the top)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("using the HF_TOKEN secret")
except Exception:
    raise SystemExit("No HF_TOKEN secret: do the one-time Hugging Face step at the top (Add-ons -> Secrets -> HF_TOKEN), then run again.")
q = dict(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=DTYPE)
transformer = FluxTransformer2DModel.from_pretrained(MODEL, subfolder="transformer", quantization_config=DiffBnb(**q), torch_dtype=DTYPE)
text_encoder_2 = T5EncoderModel.from_pretrained(MODEL, subfolder="text_encoder_2", quantization_config=TfBnb(**q), torch_dtype=DTYPE)
pipe = FluxPipeline.from_pretrained(MODEL, transformer=transformer, text_encoder_2=text_encoder_2, torch_dtype=DTYPE)
pipe.enable_model_cpu_offload()
pipe.vae.enable_tiling()
gc.collect(); torch.cuda.empty_cache()
print("model ready")

In [ ]:
# 5) Make the pictures (about 1 minute each on a T4). Already-made files are skipped, so a re-run continues where it stopped.
import json, time, datetime
import numpy as np
from PIL import Image
OUT = "/kaggle/working/art_kids_v3"
os.makedirs(OUT, exist_ok=True)
manifest = []
t_start = time.time(); n = 0
for name, what, style, jobs in PICTURES:
    prompt = f"{what}, {style}"
    for orient, seed in jobs:
        n += 1
        w, h, final = SIZES[orient]
        fn = f"kids/{name}_{orient}_s{seed}.png"
        path = os.path.join(OUT, fn)
        os.makedirs(os.path.dirname(path), exist_ok=True)
        if not os.path.exists(path):
            t = time.time()
            img = pipe(prompt, width=w, height=h, num_inference_steps=4, guidance_scale=0.0, max_sequence_length=256,
                       generator=torch.Generator("cpu").manual_seed(seed)).images[0]
            if np.asarray(img).mean() < 3:
                print("WARNING: black picture for", fn, "- tell Claude")
            img.resize(final, Image.LANCZOS).save(path, optimize=True)
            print(f"[{n}/{TOTAL}] made {fn} in {time.time() - t:.0f}s")
        manifest.append({"file": fn, "name": name, "orientation": orient, "seed": seed, "size": list(final),
                         "made_at": [w, h], "prompt": prompt, "model": MODEL, "steps": 4,
                         "licence": "Apache-2.0 (FLUX.1-schnell)", "date": datetime.date.today().isoformat()})
json.dump(manifest, open(os.path.join(OUT, "manifest.json"), "w"), indent=1)
print(f"done: {len(manifest)} pictures in {(time.time() - t_start) / 60:.0f} min")

In [ ]:
# 6) Contact sheet to look at, and the zip to download (Output -> art_kids_v3.zip)
from PIL import ImageDraw
thumbs = []
for m in manifest:
    im = Image.open(os.path.join(OUT, m["file"])).convert("RGB")
    im.thumbnail((270, 270)); tile = Image.new("RGB", (280, 300), "white"); tile.paste(im, ((280 - im.width) // 2, 5))
    ImageDraw.Draw(tile).text((6, 282), m["file"].split("/")[-1][:40], fill="black"); thumbs.append(tile)
cols = 9; rows = (len(thumbs) + cols - 1) // cols
sheet = Image.new("RGB", (cols * 280, rows * 300), "white")
for i, t in enumerate(thumbs): sheet.paste(t, ((i % cols) * 280, (i // cols) * 300))
sheet.save(os.path.join(OUT, "contact_sheet.jpg"), quality=85)
import shutil
shutil.make_archive("/kaggle/working/art_kids_v3", "zip", OUT)
print("zip:", round(os.path.getsize("/kaggle/working/art_kids_v3.zip") / 2**20, 1), "MB")
print("NOW: download art_kids_v3.zip from Output (right-hand panel) BEFORE stopping the session.")
sheet